## **Build a ReAct Agent**

You're a software engineer on a mission: build an AI agent that doesn't just respond—it thinks. In this lab, you'll step into the role of an AI architect, designing a smart assistant that solves tough problems by reasoning through them and taking purposeful actions.

Using the ReAct (Reasoning + Acting) framework, you'll teach your agent to think step by step, consult tools like search engines or calculators, and adapt on the fly. It’s not just about answers—it’s about how the agent gets there.

By the end of the lab, your AI will face a mystery that can’t be solved with knowledge alone. It will need logic, resourcefulness, and the ability to act—just like you, the engineer who built it.

## What is ReAct?

**ReAct** stands for **Reasoning + Acting**. It's a framework that combines:

1. **Reasoning**: The agent thinks through problems step by step, maintaining an internal dialogue about what it needs to do.
2. **Acting**: The agent can use external tools (search engines, calculators, APIs) to gather information or perform actions.
3. **Observing**: The agent processes the results from its actions and incorporates them into its reasoning.

This creates a powerful loop: **Think → Act → Observe → Think → Act → ...**

### Why ReAct Matters

Traditional language models are limited by their training data cutoff and can't access real-time information. ReAct agents overcome this by:
- Accessing current information through web searches
- Performing calculations with specialized tools
- Breaking down complex problems into manageable steps
- Adapting their approach based on intermediate results


In [2]:
from langchain_community.tools.tavily_search import TavilySearchResults
import os
import dotenv

In [3]:
dotenv.load_dotenv()
taily_key = os.getenv("travily_api_key")
groq_key = os.getenv("groq_api")

In [6]:
tavily_search = TavilySearchResults(tavily_api_key=taily_key, max_results=3)

In [7]:
from langchain_groq import ChatGroq

In [8]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=groq_key
)
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000021D02087C10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021D04AC9E10>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [9]:
tavily_search.invoke("What is the weather in Colombo now?")

[{'title': 'Weather - Colombo - 14-Day Forecast & Rain | Ventusky',
  'url': 'https://www.ventusky.com/6.94;79.85',
  'content': "Temperature\n Feels like temperature\n Precipitation\n Radar\n Satellite\n Wind speed\n Wind gusts\n Clouds\n Air pressure\n CAPE\n Snow cover\n Humidity\n Sea\n Air quality\n\nWind Animation   \n ECMWF, GFS, ICON, GEM\n\nApp Now Offers Weather Forecasts Tailored to Your Outdoor Activities Forecast\n\nWorld / Sri Lanka / Colombo\n\n6°56'N / 79°51'E / Altitude 39 ft / 06:42 07/16/2026, Asia/Colombo (UTC+5)\n\n Current Weather\n Forecast\n Sun and Moon\n\n|  |  |\n --- |\n| overcast | 82 °F |\n|  |\n| Wind speed   2 mph |\n\n|  |  |\n --- |\n| Precipitation (9 hr.) | 0 inch |\n| Air pressure | 29.8 inHg |\n| Visibility | 12 mi |\n| Clouds | 90 % |\n\nWeather report from station Colombo, Distance: 3 mi (05:30 07/16/2026)\n\nOpen Radar Map\n\n## Weather for the next 24 hours",
  'score': 0.8724455},
 {'title': 'Colombo, Weather for July, Sri Lanka',
  'url': 'ht

In [10]:
from langchain_core.tools import tool

In [12]:
@tool
def clothing_recommendation(weather:str) -> str:
    
    """Returns clothing recommendation based on the given weather condition
    
    This function examines the input string for specific keywords or temperature indicators 
    (e.g., "snow", "freezing", "rain", "85°F") to suggest appropriate attire. It handles 
    common weather conditions like snow, rain, heat, and cold by providing simple and practical 
    clothing advice.
    
    Args:
    - weather (str): A brief description of the weather (e.g., "Overcast, 64.9°F")
    
    Return:
    - str: A string with clothing recommendations suitable for the weather
    """
    
    weather = weather.lower()
    
    if "snow" in weather or "freezig" in weather:
        return "Wear a heavy coat, gloves, and boots."
    elif "rain" in weather or "wet" in weather:
        return "Bring a raincoat and waterproof shoes."
    elif "hot" in weather or "85" in weather:
        return "T-shirt, shorts, and sunscreen recommended."
    elif "cold" in weather or "50" in weather:
        return "Wear a warm jacket or sweater."
    else:
        return "A light jacket should be fine."


In [11]:
@tool
def search_tool(query:str):
    """
    Take a string as a input and search the web for information using Tavily API

    Args:
        query (str): The search query string
    
    Returns:
        str: Search results related to query
    """
    
    results = tavily_search.invoke(query)
    return results

In [13]:
tools = [search_tool, clothing_recommendation]

tools_map = {
    tool.name:tool for tool in tools
}